<a href="https://colab.research.google.com/github/Goseungeun/2026_BigData_Analyst/blob/main/Part2/Chapter5_multiclass_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 문제 정의
- 개인 신용 관련 정보 데이터 -> 신용 등급 예측
- 평가기준 : macro-f1
- label(target) : 신용등급 (Credit_Score):1,2,3
- 제출방식 : test 데이터로 예측한 class 1개 컬럼만 csv로 제출 (컬럼명 : pred, 파일명 : result.csv)

In [1]:
import pandas as pd

train = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert/main/part2/ch5/train.csv")
test = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert/main/part2/ch5/test.csv")

## EDA

In [2]:
train.shape, test.shape

((10000, 21), (10000, 20))

In [3]:
train.head()

,Delay_from_due_date,Num_of_Delayed_Payment,Num_Credit_Inquiries,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Amount_invested_monthly,Monthly_Balance,Credit_Score,Credit_Mix,...,Age,Annual_Income,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Monthly_Inhand_Salary,Changed_Credit_Limit,Outstanding_Debt,Total_EMI_per_month
0,56.0,16.0,11.0,35.598217,120.0,Yes,229.093478,252.385965,1,Bad,...,15.0,36597.56,8.0,10.0,29.0,5.0,3143.796667,22.49,2963.18,122.900223
1,49.0,23.0,12.0,25.553106,120.0,Yes,104.613906,219.105944,1,Bad,...,28.0,32057.30,9.0,8.0,16.0,7.0,2606.441667,1.40,1327.26,164.859426
2,34.0,20.0,6.0,40.039954,174.0,Yes,338.626965,251.265589,1,Bad,...,46.0,75868.80,6.0,10.0,32.0,7.0,6074.400000,3.60,1432.71,297.547446
3,21.0,13.0,8.0,25.711678,143.0,NM,116.816864,259.927960,2,Standard,...,46.0,17092.69,7.0,3.0,19.0,7.0,1695.390833,16.40,1417.06,62.794260
4,19.0,13.0,6.0,39.140463,138.0,Yes,87.262887,626.212330,1,Bad,...,45.0,81471.96,6.0,6.0,25.0,5.0,6763.330000,27.09,2679.69,202.857783


In [5]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 21 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Delay_from_due_date       10000 non-null  float64
 1   Num_of_Delayed_Payment    10000 non-null  float64
 2   Num_Credit_Inquiries      10000 non-null  float64
 3   Credit_Utilization_Ratio  10000 non-null  float64
 4   Credit_History_Age        10000 non-null  float64
 5   Payment_of_Min_Amount     10000 non-null  object 
 6   Amount_invested_monthly   10000 non-null  float64
 7   Monthly_Balance           10000 non-null  float64
 8   Credit_Score              10000 non-null  int64  
 9   Credit_Mix                10000 non-null  object 
 10  Payment_Behaviour         10000 non-null  object 
 11  Age                       10000 non-null  float64
 12  Annual_Income             10000 non-null  float64
 13  Num_Bank_Accounts         10000 non-null  float64
 14  Num_Cre

In [7]:
train.describe(include="O")

,Payment_of_Min_Amount,Credit_Mix,Payment_Behaviour
count,10000,10000,10000
unique,3,3,6
top,Yes,Standard,Low_spent_Small_value_payments
freq,5269,4591,3416


In [8]:
test.describe(include="O")

,Payment_of_Min_Amount,Credit_Mix,Payment_Behaviour
count,10000,10000,10000
unique,3,3,6
top,Yes,Standard,Low_spent_Small_value_payments
freq,5167,4590,3498


In [10]:
train.isnull().sum().sum()

np.int64(0)

In [11]:
test.isnull().sum().sum()

np.int64(0)

In [12]:
train['Credit_Score'].value_counts()

,count
Credit_Score,
2,5237
1,2978
3,1785


## Data Preprocessing

In [13]:
target = train.pop('Credit_Score')
cols = train.columns[train.dtypes == object]
cols

Index(['Payment_of_Min_Amount', 'Credit_Mix', 'Payment_Behaviour'], dtype='object')

In [15]:
for col in cols:
  train[col] = train[col].astype('category')
  test[col] = test[col].astype('category')

In [16]:
train[cols].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 3 columns):
 #   Column                 Non-Null Count  Dtype   
---  ------                 --------------  -----   
 0   Payment_of_Min_Amount  10000 non-null  category
 1   Credit_Mix             10000 non-null  category
 2   Payment_Behaviour      10000 non-null  category
dtypes: category(3)
memory usage: 29.9 KB


In [17]:
train[cols].head()

,Payment_of_Min_Amount,Credit_Mix,Payment_Behaviour
0,Yes,Bad,High_spent_Medium_value_payments
1,Yes,Bad,High_spent_Small_value_payments
2,Yes,Bad,Low_spent_Large_value_payments
3,NM,Standard,High_spent_Small_value_payments
4,Yes,Bad,High_spent_Medium_value_payments


## Split Data

In [18]:
from sklearn.model_selection import train_test_split
X_train,X_val,y_train,y_val = train_test_split(train,target,test_size=0.2,random_state=0)
X_train.shape,X_val.shape,y_train.shape,y_val.shape

((8000, 20), (2000, 20), (8000,), (2000,))

## Train & Validation

In [19]:
from sklearn.metrics import f1_score, accuracy_score
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier()
rf.fit(X_train,y_train)

ValueError: could not convert string to float: 'Yes'

In [20]:
import lightgbm as lgb
lgbmc = lgb.LGBMClassifier(random_state=0,verbose=-1)
lgbmc.fit(X_train,y_train)
pred = lgbmc.predict(X_val)
pred

array([2, 2, 2, ..., 3, 2, 2])

In [21]:
accuracy = accuracy_score(y_val,pred)
f1 = f1_score(y_val,pred,average='macro')

print(accuracy , f1)

0.6985 0.6777379561595467


## Predict & Create Result

In [22]:
pred = lgbmc.predict(test)

result = pd.DataFrame({"pred":pred})
result.to_csv("result.csv",index=False)

In [23]:
pd.read_csv("result.csv")

,pred
0,2
1,1
2,1
3,2
4,1
...,...
9995,2
9996,2
9997,1
9998,1
